# Week 7: Hello GIS!

Welcome to Python for GIS. This notebook introduces Python workflows that mirror what you did in QGIS during Weeks 1-6.

**What you'll learn:**
1. How Python notebooks work (cells, execution, outputs)
2. How file paths work differently in Colab vs local Jupyter
3. How to load and visualize spatial data with GeoPandas
4. How Python operations map to QGIS operations you already know

---

## Python ↔ QGIS Comparison

| QGIS Operation | Python Equivalent |
|----------------|-------------------|
| Layer > Add Vector Layer | `gpd.read_file("data.shp")` |
| Open Attribute Table | `gdf.head()` or just `gdf` |
| Style layer (single symbol) | `gdf.plot(color='blue')` |
| Style layer (categorized) | `gdf.plot(column='field', categorical=True)` |
| Export > Save As | `gdf.to_file("output.gpkg")` |
| Print Layout > Export | `plt.savefig("map.png")` |

---

## How to run cells

Click on a code cell and press **Shift + Enter** to run it.
Run cells in order from top to bottom.

---

## Understanding Your Environment

**Important concept:** Where is this code actually running?

| If you opened with... | Code runs on... | Files live on... |
|----------------------|-----------------|------------------|
| **Google Colab** | Google's servers | Your Google Drive |
| **Jupyter (local)** | Your computer | Your hard drive |

**Why does this matter?**

When you write `gpd.read_file("data/my_file.geojson")`, Python looks for that file wherever the code is running.

- **Local Jupyter:** Looks on your computer → files are already there
- **Google Colab:** Looks on Google's server → your files aren't there!

**The solution:** In Colab, we "mount" (connect) your Google Drive so the code can access your files.

---

## Step 1: Set up your environment

This cell detects where you're running and installs required packages.

**What happens:**
- `sys.modules` is a dictionary of all imported Python modules
- If `'google.colab'` is in there, we're running in Colab
- `!pip install` runs a shell command to install packages

In [ ]:
# ============================================================
# STEP 1: DETECT ENVIRONMENT AND INSTALL PACKAGES
# ============================================================
# This is a common pattern you'll see in all our notebooks.
# It makes the notebook work in both Colab and local Jupyter.

import sys  # System-specific parameters and functions

# Check if we're running in Google Colab
# sys.modules is a dictionary of all currently loaded modules
# If 'google.colab' is loaded, we're in Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Running in Google Colab")
    print("Installing GIS packages (takes ~1 minute)...")
    
    # The ! prefix runs shell commands from within Python
    # pip install downloads and installs Python packages
    # -q means "quiet" - less output to keep things clean
    !pip install geopandas contextily folium -q
    
    print("Done! Packages installed.")
else:
    print("Running locally (Jupyter)")
    print("Make sure you activated your environment: conda activate intro-gis")

---

## Step 2: Set up file paths

**QGIS equivalent:** Setting up your project folder structure in Week 1.

We use the same folder structure as QGIS:
- `data/raw/` — Original data files (never modify these)
- `data/processed/` — Your analysis outputs
- `exports/` — Final maps and reports

**Why `pathlib.Path`?**
- Works on Windows (`\`) and Mac/Linux (`/`) automatically
- Clean syntax: `RAW / "filename.shp"` instead of `RAW + "/filename.shp"`

In [ ]:
# ============================================================
# STEP 2: SET UP DATA PATHS
# ============================================================
# pathlib.Path provides a clean way to work with file paths.
# It automatically handles the difference between Windows (\)
# and Mac/Linux (/) path separators.

from pathlib import Path

if IN_COLAB:
    # --------------------------------------------------------
    # COLAB: Mount Google Drive
    # --------------------------------------------------------
    # "Mounting" connects your Google Drive to the Colab computer.
    # It's like plugging in a USB drive - suddenly the computer
    # can see all your files.
    
    from google.colab import drive
    drive.mount('/content/drive')  # This prompts you to authorize
    
    # Define paths to your Google Drive folders
    # These paths point to: My Drive/intro-gis/week07/...
    RAW = Path("/content/drive/MyDrive/intro-gis/week07/data/raw")
    PROCESSED = Path("/content/drive/MyDrive/intro-gis/week07/data/processed")
    EXPORTS = Path("/content/drive/MyDrive/intro-gis/week07/exports")
    
else:
    # --------------------------------------------------------
    # LOCAL: Use relative paths
    # --------------------------------------------------------
    # Relative paths start from where the notebook is located.
    # If notebook is in /intro-gis/notebooks/, then "data/raw"
    # means /intro-gis/notebooks/data/raw/
    
    RAW = Path("data/raw")
    PROCESSED = Path("data/processed")
    EXPORTS = Path("exports")

# Create folders if they don't exist
# parents=True: create parent folders too (e.g., data/ before data/raw/)
# exist_ok=True: don't raise an error if folder already exists
for folder in [RAW, PROCESSED, EXPORTS]:
    folder.mkdir(parents=True, exist_ok=True)

print("Folder structure ready:")
print(f"  RAW (input data):     {RAW}")
print(f"  PROCESSED (outputs):  {PROCESSED}")
print(f"  EXPORTS (final maps): {EXPORTS}")
print("\nUsage examples:")
print(f"  Load data:  gpd.read_file(RAW / 'my_data.geojson')")
print(f"  Save data:  gdf.to_file(PROCESSED / 'results.gpkg')")
print(f"  Save map:   plt.savefig(EXPORTS / 'my_map.png')")

---

## Step 3: Import libraries

**What are libraries?**

Libraries are collections of code that other people wrote. Instead of writing mapping code from scratch, we use `geopandas` (built on top of hundreds of thousands of lines of code).

**Key libraries for GIS:**

| Library | What it does | QGIS equivalent |
|---------|--------------|------------------|
| `geopandas` | Spatial data manipulation | Layer operations, attribute table |
| `pandas` | Tabular data (CSV, Excel) | Joining CSV tables |
| `matplotlib` | Creating plots and maps | Print Layout |
| `shapely` | Geometry operations | Geometry tools |

**The `as` keyword:** Creates a shorter alias. `import geopandas as gpd` lets us write `gpd.read_file()` instead of `geopandas.read_file()`.

In [ ]:
# ============================================================
# STEP 3: IMPORT LIBRARIES
# ============================================================
# These import statements load external code libraries.
# We use standard abbreviations (gpd, pd, plt) that you'll
# see in tutorials and documentation everywhere.

import geopandas as gpd      # GeoDataFrames = spatial data (like shapefiles)
import pandas as pd          # DataFrames = tabular data (like CSV files)
import matplotlib.pyplot as plt  # Plotting and visualization
from shapely.geometry import Point  # Create point geometries

print("Libraries imported successfully!")
print(f"\nGeoPandas version: {gpd.__version__}")

---

## Step 4: Create sample data

**QGIS equivalent:** Week 1 - loading Natural Earth countries and seeing the attribute table.

In Python, we can create data directly in code. This is useful for:
- Learning without needing to download files
- Creating test data for scripts
- Documenting example workflows

**Key concept: GeoDataFrame**

A GeoDataFrame is like a shapefile loaded into memory:
- Each row is a feature (like a row in QGIS attribute table)
- Each column is an attribute (like a field in QGIS)
- One special column called `geometry` holds the shapes

In [ ]:
# ============================================================
# STEP 4: CREATE SAMPLE DATA (AUSTRALIAN CITIES)
# ============================================================
# This demonstrates how to create spatial data from scratch.
# In QGIS, this would be like creating a new point layer and
# manually adding features with coordinates.

# Step 4a: Define our data as a Python dictionary
# A dictionary uses key: value pairs to store data
cities_data = {
    'city': ['Sydney', 'Melbourne', 'Brisbane', 'Perth', 'Adelaide'],
    'population': [5312000, 5078000, 2514000, 2085000, 1376000],
    'longitude': [151.2093, 144.9631, 153.0251, 115.8605, 138.6007],
    'latitude': [-33.8688, -37.8136, -27.4698, -31.9505, -34.9285]
}

# Step 4b: Convert dictionary to a DataFrame (like a spreadsheet)
# pd.DataFrame() creates a table from our dictionary
df = pd.DataFrame(cities_data)

# Step 4c: Create Point geometries from coordinates
# zip() pairs up longitude and latitude values
# Point(x, y) creates a point geometry at those coordinates
# This list comprehension creates a Point for each city
geometry = [Point(lon, lat) for lon, lat in zip(df['longitude'], df['latitude'])]

# Step 4d: Create GeoDataFrame (DataFrame + geometry = spatial data)
# crs='EPSG:4326' sets the Coordinate Reference System
# EPSG:4326 = WGS84 (latitude/longitude in degrees) - same as QGIS default
cities = gpd.GeoDataFrame(df, geometry=geometry, crs='EPSG:4326')

print(f"Created GeoDataFrame with {len(cities)} cities")
print(f"CRS: {cities.crs}")
print("\n--- This is like opening the attribute table in QGIS ---")

# Display the data (like viewing the attribute table)
cities

---

## Step 5: Load data from a URL

**QGIS equivalent:** Layer > Add Vector Layer > select a file

GeoPandas can load data from:
- Local files: `gpd.read_file("path/to/file.shp")`
- URLs: `gpd.read_file("https://example.com/data.geojson")`
- Zipped files: `gpd.read_file("data.zip")`

This is the same Natural Earth data you used in Week 1!

In [ ]:
# ============================================================
# STEP 5: LOAD DATA FROM URL
# ============================================================
# gpd.read_file() is incredibly versatile - it can read:
# - Shapefiles (.shp)
# - GeoJSON (.geojson)
# - GeoPackage (.gpkg)
# - And many more formats!
#
# QGIS equivalent: Layer > Add Layer > Add Vector Layer

# Load Natural Earth countries (same data as QGIS Week 1!)
# This downloads a ~4MB file from the internet
print("Loading world boundaries from Natural Earth...")
print("(This is the same ne_110m_admin_0_countries you used in QGIS Week 1)")
print("")

world = gpd.read_file(
    "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
)

print(f"Loaded {len(world)} countries")
print(f"CRS: {world.crs}")
print(f"\nColumns (like fields in QGIS attribute table):")
print(f"  {list(world.columns[:10])}...")  # Show first 10 columns

# Filter to just Australia
# This is like: Select by Expression > "NAME" = 'Australia'
australia = world[world['NAME'] == 'Australia']

print(f"\nFiltered to: {australia['NAME'].values[0]}")

---

## Step 6: Explore the data

**QGIS equivalent:** Using the Identify tool to click on features and see their attributes.

Useful methods for exploring data:
- `.head()` - View first few rows
- `.info()` - Column types and memory usage
- `.describe()` - Statistics for numeric columns
- `.columns` - List all column names

In [ ]:
# ============================================================
# STEP 6: EXPLORE THE DATA
# ============================================================
# These methods help you understand your data before analysis.
# QGIS equivalent: Opening the attribute table and sorting/filtering

# .head(n) shows the first n rows (default is 5)
# Like scrolling to the top of the attribute table
print("First 3 countries:")
print(world[['NAME', 'CONTINENT', 'POP_EST']].head(3))

print("\n" + "="*50)

# Sort by population (like clicking a column header in QGIS)
# ascending=False means largest first
print("\nTop 5 countries by population:")
top_5 = world.nlargest(5, 'POP_EST')[['NAME', 'POP_EST', 'CONTINENT']]
print(top_5.to_string(index=False))

print("\n" + "="*50)

# Count features by continent
# Like: Vector > Analysis Tools > Statistics by Categories
print("\nCountries per continent:")
print(world['CONTINENT'].value_counts())

---

## Step 7: Create your first map!

**QGIS equivalent:** 
- Adding layers to the map canvas
- Applying symbology (colors, sizes)
- Using Print Layout to add title and export

The `.plot()` method is the workhorse for quick visualizations.

| Python parameter | QGIS equivalent |
|------------------|------------------|
| `color='green'` | Single Symbol > Fill color |
| `edgecolor='black'` | Single Symbol > Stroke color |
| `markersize=50` | Single Symbol > Size |
| `column='field'` | Categorized/Graduated symbology |
| `legend=True` | Add legend in Print Layout |

In [ ]:
# ============================================================
# STEP 7: CREATE YOUR FIRST MAP
# ============================================================
# This creates a visualization similar to what you'd see in
# QGIS after adding layers and styling them.
#
# Key concepts:
# - fig, ax = plt.subplots() creates a figure (the canvas) and axes (the plot area)
# - .plot(ax=ax) tells each layer to draw on the same axes
# - Layers are drawn in order - later layers appear on top

# Create figure and axes
# figsize=(width, height) in inches
fig, ax = plt.subplots(figsize=(12, 8))

# Layer 1: Australia boundary (polygon layer)
# QGIS: Add Vector Layer > style with green fill
australia.plot(
    ax=ax,                    # Draw on our axes
    color='lightgreen',       # Fill color
    edgecolor='darkgreen',    # Outline color
    linewidth=1               # Outline thickness
)

# Layer 2: Cities (point layer)
# QGIS: Add Vector Layer > style points with graduated size
# markersize based on population (larger cities = larger points)
cities.plot(
    ax=ax,
    color='red',
    markersize=cities['population'] / 15000,  # Scale size by population
    zorder=5  # zorder controls layer order (higher = on top)
)

# Add labels (like enabling labels in QGIS Layer Styling)
# We loop through each city and place text at its coordinates
for idx, row in cities.iterrows():
    ax.annotate(
        row['city'],                           # The text to display
        xy=(row['longitude'], row['latitude']), # Position
        xytext=(5, 5),                         # Offset in pixels
        textcoords='offset points',            # Offset type
        fontsize=10
    )

# Style the map (like setting project properties in QGIS)
ax.set_title('Australian Capital Cities', fontsize=14)
ax.set_xlim(110, 160)  # Longitude range
ax.set_ylim(-45, -10)  # Latitude range
ax.set_facecolor('lightblue')  # Background color (ocean)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

# Display the map
plt.tight_layout()
plt.show()

print("\nYour first Python map!")
print("Compare this to the map you made in QGIS Week 1.")

---

## Step 8: Save your outputs

**QGIS equivalent:**
- Right-click layer > Export > Save Features As (for data)
- Print Layout > Export as Image/PDF (for maps)

**File formats:**

| Format | Extension | Best for |
|--------|-----------|----------|
| GeoPackage | `.gpkg` | All-purpose, recommended |
| GeoJSON | `.geojson` | Web maps, small files |
| Shapefile | `.shp` | Legacy compatibility |
| PNG | `.png` | Maps with transparency |
| PDF | `.pdf` | Print-ready maps |

In [ ]:
# ============================================================
# STEP 8: SAVE YOUR OUTPUTS
# ============================================================
# Good practice: Save processed data so you can reuse it later.
# This is especially important when analysis takes a long time.

# Save the cities GeoDataFrame as GeoJSON
# QGIS: Right-click layer > Export > Save Features As > GeoJSON
cities_output = PROCESSED / "australian_cities.geojson"
cities.to_file(cities_output, driver='GeoJSON')
print(f"Cities saved to: {cities_output}")

# Save the Australia boundary as GeoPackage
# GeoPackage (.gpkg) is the recommended format - it's like a database
australia_output = PROCESSED / "australia_boundary.gpkg"
australia.to_file(australia_output, driver='GPKG')
print(f"Australia boundary saved to: {australia_output}")

# Re-create the map and save it as an image
# QGIS: Print Layout > Export as Image
fig, ax = plt.subplots(figsize=(12, 8))
australia.plot(ax=ax, color='lightgreen', edgecolor='darkgreen', linewidth=1)
cities.plot(ax=ax, color='red', markersize=cities['population']/15000, zorder=5)
for idx, row in cities.iterrows():
    ax.annotate(row['city'], xy=(row['longitude'], row['latitude']),
                xytext=(5, 5), textcoords='offset points', fontsize=10)
ax.set_title('Australian Capital Cities', fontsize=14)
ax.set_xlim(110, 160)
ax.set_ylim(-45, -10)
ax.set_facecolor('lightblue')

# Save the figure
# dpi=300 means high resolution (good for printing)
# bbox_inches='tight' removes extra whitespace
map_output = EXPORTS / "week07_australian_cities.png"
fig.savefig(map_output, dpi=300, bbox_inches='tight')
plt.close()  # Close the figure to free memory

print(f"Map saved to: {map_output}")
print("\n" + "="*50)
print("\nTo load these files in future notebooks:")
print(f"  cities = gpd.read_file(PROCESSED / 'australian_cities.geojson')")
print(f"  australia = gpd.read_file(PROCESSED / 'australia_boundary.gpkg')")

---

## Try it yourself

**Exercise:** Add Darwin and Hobart to the cities data.

| City | Longitude | Latitude | Population |
|------|-----------|----------|------------|
| Darwin | 130.8456 | -12.4634 | 147,000 |
| Hobart | 147.3272 | -42.8821 | 238,000 |

**Hint:** You can create a new GeoDataFrame and combine them using `pd.concat()`

In [ ]:
# ============================================================
# EXERCISE: ADD MORE CITIES
# ============================================================
# Uncomment the code below and fill in the values

# new_cities_data = {
#     'city': ['Darwin', 'Hobart'],
#     'population': [147000, 238000],
#     'longitude': [130.8456, 147.3272],
#     'latitude': [-12.4634, -42.8821]
# }
# 
# new_df = pd.DataFrame(new_cities_data)
# new_geometry = [Point(lon, lat) for lon, lat in zip(new_df['longitude'], new_df['latitude'])]
# new_cities = gpd.GeoDataFrame(new_df, geometry=new_geometry, crs='EPSG:4326')
# 
# # Combine with existing cities
# all_cities = pd.concat([cities, new_cities], ignore_index=True)
# 
# # Plot to verify
# fig, ax = plt.subplots(figsize=(12, 8))
# australia.plot(ax=ax, color='lightgreen', edgecolor='darkgreen')
# all_cities.plot(ax=ax, color='red', markersize=50)
# for idx, row in all_cities.iterrows():
#     ax.annotate(row['city'], xy=(row['longitude'], row['latitude']),
#                 xytext=(5, 5), textcoords='offset points')
# ax.set_xlim(110, 160)
# ax.set_ylim(-45, -10)
# ax.set_facecolor('lightblue')
# plt.show()
# 
# print(f"Now have {len(all_cities)} cities")

---

## Summary: Python ↔ QGIS Mapping

| What you learned | Python | QGIS Week 1 equivalent |
|------------------|--------|------------------------|
| Set up workspace | `Path("data/raw")` | Creating folder structure |
| Load data | `gpd.read_file()` | Layer > Add Vector Layer |
| View attributes | `gdf.head()` | Open Attribute Table |
| Filter features | `gdf[gdf['NAME'] == 'X']` | Select by Expression |
| Style layer | `.plot(color=...)` | Symbology panel |
| Add labels | `ax.annotate()` | Enable labels |
| Save data | `gdf.to_file()` | Export > Save Features As |
| Export map | `fig.savefig()` | Print Layout > Export |

### Key concepts

1. **GeoDataFrame** = Python's version of a vector layer (shapefile)
2. **CRS** = Coordinate Reference System, same as in QGIS
3. **Geometry column** = Stores the shapes (points, lines, polygons)
4. **`.plot()`** = Quick way to visualize spatial data

---

## What's next?

**Week 8** will cover vector workflows in Python:
- Loading your own data files
- Spatial joins (like Join Attributes by Location)
- Point-in-polygon counts (like Count Points in Polygon)
- Creating choropleth maps

This mirrors what you did in **QGIS Week 3** (Vector Analysis & Attribute Joins).

---

**Save your work:**
- Colab: `File > Save a copy in Drive`
- Local: `Ctrl+S` or `Cmd+S`